# Hurst-ADF-KPSS 三重趋势验证器演示

本notebook演示如何使用三重趋势验证器分析Fusion Bar的趋势性。

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
from jesse import helpers, research
from src.bars.fusion.demo import DemoBar
from research.hurst_adf_kpss import TrendValidator

## 1. 获取K线数据并生成Fusion Bar

In [ ]:
# 获取BTC 1分钟K线
START = "2024-01-01"
END = "2024-12-01"

_, candles = research.get_candles(
    "Binance Perpetual Futures",
    "BTC-USDT",
    "1m",
    helpers.date_to_timestamp(START),
    helpers.date_to_timestamp(END),
    warmup_candles_num=0,
    caching=False,
    is_for_jesse=False,
)

# 过滤零成交量
candles = candles[candles[:, 5] > 0]
print(f"原始K线数量: {len(candles)}")

In [ ]:
# 生成Fusion Bar
bar_container = DemoBar(clip_r=0, threshold=1.399)
bar_container.update_with_candles(candles)
fusion_bars = bar_container.get_fusion_bars()

print(f"Fusion Bar数量: {len(fusion_bars)}")
print(f"压缩比: {len(fusion_bars) / len(candles) * 100:.2f}%")

## 2. 三重趋势验证

In [ ]:
# 创建验证器 (窗口大小=40, 步长=5)
validator = TrendValidator(window_size=40, step=5)

# 执行验证
results = validator.validate(fusion_bars)
results.head(10)

In [ ]:
# 汇总统计
summary = validator.summarize(results)
print("汇总统计:")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## 3. 评分分布

In [ ]:
# 评分分布
score_counts = results['score'].value_counts().sort_index()
print("评分分布:")
for score, count in score_counts.items():
    pct = count / len(results) * 100
    print(f"  {score}分: {count} ({pct:.1f}%)")

In [ ]:
# 趋势类型分布
trend_counts = results['trend_type'].value_counts()
print("趋势类型分布:")
for trend, count in trend_counts.items():
    pct = count / len(results) * 100
    print(f"  {trend}: {count} ({pct:.1f}%)")

## 4. 多窗口对比

In [ ]:
# 对比不同窗口大小
window_sizes = [20, 40, 60, 80]
summaries = []

for ws in window_sizes:
    v = TrendValidator(window_size=ws, step=5)
    r = v.validate(fusion_bars)
    s = v.summarize(r)
    summaries.append(s)
    print(f"窗口{ws}: 平均分={s['mean_score']:.2f}, 5分占比={s['high_score_ratio']*100:.1f}%")